# Variance Partitioning — Protein Level (variancePartition)

Quantifies how much of the variability in protein-level log₂ intensities is explained by technical versus biological factors using the `variancePartition` framework (Hoffman & Schadt 2016). This complements the precursor-level scplainer analysis in `01_scplainer.ipynb` by operating at the protein level with a mixed-effects model that explicitly accounts for the nested plate/lab structure.

**Key differences from `01_scplainer.ipynb`:**

| Aspect | scplainer (`01_scplainer.ipynb`) | variancePartition (this notebook) |
|---|---|---|
| Feature level | Precursor | Protein |
| Model framework | scplainer linear model | `lme4` mixed-effects via `fitExtractVarPartModel` |
| Random effects | No (all fixed) | Yes — plate, date, plate × position |
| Injection order | Not modelled | Modelled as continuous fixed effect |
| Missing values | Excluded (Diameter) / n/p filter | Row-mean imputed after protein filter |

**Modelling design — formula covariates:**

| Term | Type | Role |
|---|---|---|
| `(1\|plate_id)` | Random | Plate-level batch effect |
| `(0 + X_pos \| plate_id)` | Random slope | Plate-specific injection-position gradient |
| `(1\|plate_id:date)` | Random (nested) | Acquisition-day effect within plate |
| `injection_order_z` | Fixed | Within-plate run order (continuous, z-scored) |
| `(1\|source name)` | Random | Cell type — biological contrast of interest |
| `Diameter_z` | Fixed | Cell diameter from cellenONE (continuous, z-scored) |
| `n_proteins_z` | Fixed | Number of identified proteins — normalisation proxy |
| `log2_median_intensity_z` | Fixed | Median log₂ intensity — normalisation proxy |

**Inputs** (`Data/AnnData/out/`)

| File | Description |
|---|---|
| `adata_protein.h5ad` | Protein-level log₂ intensities in `layers["X_log2"]`; `obs` carries all sample metadata |

**Outputs** (`Data/variancePartition/`)

| File | Description |
|---|---|
| `vp_fit.rds` | variancePartition result — full model |
| `vp_fit_norm.rds` | variancePartition result — after normalisation regression |
| `vp_fit_batch.rds` | variancePartition result — after full batch regression |
| `variance_fractions_full.csv` | Per-protein variance fractions, full model |

---

## Imports

In [ ]:
library(reticulate)
use_condaenv("r44_varpart", required = TRUE)
py_config()
library(anndata)

pkgs <- c(
    "anndata", "reticulate",
    "variancePartition", "BiocParallel",
    "limma", "lme4", "reformulas",
    "ggplot2", "patchwork", "pheatmap",
    "RColorBrewer", "scales",
    "tidyr", "dplyr", "lubridate"
)

cat("Package check\n")
cat(strrep("─", 40), "\n")
for (p in pkgs) {
    status  <- requireNamespace(p, quietly = TRUE)
    version <- if (status) as.character(packageVersion(p)) else "MISSING"
    cat(sprintf("  %-25s %s\n", p, version))
}
cat("\n")

suppressPackageStartupMessages({
    invisible(lapply(pkgs, library, character.only = TRUE))
})

# Parallel backend
if (.Platform$OS.type == "windows") {
    register(SnowParam(workers = 64))
} else {
    register(MulticoreParam(workers = 64))
}

## 0. Path Configuration

Output directories are resolved relative to the notebook location and created automatically if absent. Both can be overridden via environment variables for CI or cross-machine runs.

| Variable | Default | Environment override |
|---|---|---|
| `ROOT_OUT` | `../../Data` | `SCI_DATA_DIR` |
| `FIGURES_OUT` | `../../Figures` | `SCI_FIGURES_DIR` |

Model RDS files are written to `ROOT_OUT/variancePartition/` and read back from there in later sections. AnnData objects are read from `ROOT_OUT/AnnData/out/`, produced by the QC pipeline.

In [ ]:
NOTEBOOK_DIR <- normalizePath(getwd())
ROOT_OUT     <- normalizePath(file.path(NOTEBOOK_DIR, "../../Data"))
ROOT_OUT     <- Sys.getenv("SCI_DATA_DIR",    unset = ROOT_OUT)

FIGURES_OUT  <- normalizePath(file.path(NOTEBOOK_DIR, "../../Figures"))
FIGURES_OUT  <- Sys.getenv("SCI_FIGURES_DIR", unset = FIGURES_OUT)

ANNDATA_IN  <- file.path(ROOT_OUT, "AnnData", "out")
MODELS_OUT  <- file.path(ROOT_OUT, "variancePartition")
FIG_OUT     <- file.path(FIGURES_OUT, "Figure_6", "VariancePartition")

for (d in c(ANNDATA_IN, MODELS_OUT, FIG_OUT)) {
    dir.create(d, recursive = TRUE, showWarnings = FALSE)
}

cat(sprintf("Data directory    : %s\n", ROOT_OUT))
cat(sprintf("Figures directory : %s\n", FIGURES_OUT))
cat(sprintf("Models directory  : %s\n", MODELS_OUT))
cat(sprintf("AnnData input     : %s\n", ANNDATA_IN))

## 1. Load Data

Reads the protein-level AnnData object and extracts the log₂-transformed intensity matrix from `layers["X_log2"]`. The matrix is transposed to proteins × cells for compatibility with `variancePartition` and `limma`. Sample metadata from `obs` is extracted as `meta` and carries all covariates used in the model formula.

> The AnnData is expected to contain only cells that passed the upstream QC pipeline. No additional cell filtering is applied here.

In [ ]:
adata    <- read_h5ad(file.path(ANNDATA_IN, "adata_protein.h5ad"))
expr_mat <- t(as.matrix(adata$layers["X_log2"]))   # proteins x cells
meta     <- as.data.frame(adata$obs)

cat(sprintf("Proteins x cells : %d x %d\n", nrow(expr_mat), ncol(expr_mat)))
cat(sprintf("NA values        : %d  (%.1f%%)\n",
            sum(is.na(expr_mat)),
            100 * mean(is.na(expr_mat))))
cat("\nMetadata columns:\n")
print(colnames(meta))

## 2. Parse Variables

Three preparatory steps derive the covariates needed for the mixed-effects model:

1. **Injection order** — computed from the gradient-start timestamp per plate. Cells are ranked within plate and rescaled to [0, 1]. Cells with missing timestamps receive the within-plate median; any remaining `NA` receives 0.5 (mid-run position).
2. **Column definitions and type coercion** — column-name constants are defined once here and referenced throughout; categorical variables are converted to factors; continuous covariates are z-scored.
3. **Missing value audit** — `Diameter` (~11%) is the only covariate with structural missingness. Cells without a `Diameter` measurement are excluded from modelling (drops from the `meta` and `expr_mat` objects at the end of this section).

### 2.1 Injection Order

Within each plate, cells are ranked by their gradient-start datetime to reconstruct injection order. The rank is normalised to [0, 1] so the covariate is on a comparable scale across plates of different size. Three fallback rules handle missing timestamps:

1. Cells with a valid datetime → rank-based normalised order within plate.
2. Cells with `NA` datetime but other plate members have datetimes → within-plate median.
3. All other remaining `NA` → 0.5 (conservative mid-run assumption).

In [ ]:
meta <- meta %>%
    mutate(datetime_parsed = ymd_hms(`comment[Gradient time]_datetime`)) %>%
    group_by(plate_id) %>%
    mutate(
        injection_rank  = rank(datetime_parsed, ties.method = "average", na.last = "keep"),
        n_observed      = sum(!is.na(datetime_parsed)),
        injection_order = (injection_rank - 1) / (n_observed - 1)
    ) %>%
    ungroup() %>%
    mutate(injection_order = ifelse(is.nan(injection_order), 0.5, injection_order)) %>%
    group_by(plate_id) %>%
    mutate(
        injection_order = ifelse(
            is.na(injection_order),
            median(injection_order, na.rm = TRUE),
            injection_order
        )
    ) %>%
    ungroup() %>%
    mutate(injection_order = ifelse(is.na(injection_order), 0.5, injection_order))

cat("injection_order summary:\n")
print(summary(meta$injection_order))
cat(sprintf("NAs remaining: %d\n", sum(is.na(meta$injection_order))))

meta %>%
    group_by(plate_id) %>%
    summarise(
        n         = n(),
        n_imputed = sum(is.na(ymd_hms(`comment[Gradient time]_datetime`))),
        min_order = min(injection_order),
        max_order = max(injection_order),
        .groups   = "drop"
    ) %>%
    print()

### 2.2 Column Definitions and Type Coercion

Column-name constants are defined once here so that formula construction and subsetting throughout the notebook reference the same strings. Categorical columns are converted to `factor` (required by `lme4` for random effects); continuous covariates are z-scored to put fixed-effect coefficients on a common scale and to improve numerical stability of the mixed-effects solver.

In [ ]:
# Column-name constants
COL_CELL_TYPE       <- "source name"
COL_INSTRUMENT      <- "instrument_type"
COL_X_POSITION      <- "comment[X position]"
COL_Y_POSITION      <- "comment[Y position]"
COL_DIAMETER        <- "Diameter"
COL_N_PROTEINS      <- "n_proteins"
COL_LOG2_MED_INT    <- "log2_median_intensity"
COL_PLATE           <- "plate_id"
COL_DATE            <- "comment[Gradient time]_date"
COL_INJ_ORDER       <- "injection_order"

required_cols <- c(COL_CELL_TYPE, COL_INSTRUMENT, COL_X_POSITION, COL_Y_POSITION,
                   COL_DIAMETER, COL_N_PROTEINS, COL_LOG2_MED_INT,
                   COL_PLATE, COL_DATE, COL_INJ_ORDER)
missing_cols <- setdiff(required_cols, colnames(meta))
if (length(missing_cols) > 0) {
    stop("Missing columns in obs: ", paste(missing_cols, collapse = ", "))
}
cat("All required columns present.\n\n")

# Factors
meta[[COL_CELL_TYPE]]  <- factor(meta[[COL_CELL_TYPE]])
meta[[COL_INSTRUMENT]] <- factor(meta[[COL_INSTRUMENT]])
meta[[COL_PLATE]]      <- factor(meta[[COL_PLATE]])
meta[[COL_DATE]]       <- factor(meta[[COL_DATE]])

# Continuous
meta[[COL_DIAMETER]]     <- as.numeric(meta[[COL_DIAMETER]])
meta[[COL_N_PROTEINS]]   <- as.numeric(meta[[COL_N_PROTEINS]])
meta[[COL_LOG2_MED_INT]] <- as.numeric(meta[[COL_LOG2_MED_INT]])

# Z-scores (used in formulas; keeps original columns intact)
meta$Diameter_z              <- scale(meta[[COL_DIAMETER]])[, 1]
meta$n_proteins_z            <- scale(meta[[COL_N_PROTEINS]])[, 1]
meta$log2_median_intensity_z <- scale(meta[[COL_LOG2_MED_INT]])[, 1]
meta$injection_order_z       <- scale(meta[[COL_INJ_ORDER]])[, 1]
meta$X_position_z            <- scale(meta[[COL_X_POSITION]])[, 1]

cat("Type summary:\n")
type_summary <- sapply(required_cols, function(v) class(meta[[v]]))
print(type_summary)

### 2.3 Missing Value Audit and Cell Exclusion

Checks all model covariates for `NA` values. `Diameter` (~11%) is the only covariate with structural missingness, caused by cells that could not be imaged by the cellenONE. Mean imputation would attenuate the estimated `Diameter` effect and bias the variance decomposition, so cells without a `Diameter` measurement are excluded from both `meta` and `expr_mat` before modelling. All cells are retained above for injection-order computation and type coercion.

> Protein-level missing values (sparse detection across cells) are handled separately in Section 5.1 via row-mean imputation, which is standard practice for `fitExtractVarPartModel`.

In [ ]:
formula_vars <- c(COL_CELL_TYPE, COL_INSTRUMENT, COL_X_POSITION,
                  COL_DIAMETER, COL_N_PROTEINS, COL_LOG2_MED_INT,
                  COL_PLATE, COL_DATE, COL_INJ_ORDER)

na_summary <- sapply(formula_vars, function(v) {
    x <- meta[[v]]
    c(n_NA = sum(is.na(x)), pct_NA = round(mean(is.na(x)) * 100, 1), n_total = length(x))
})
print(t(na_summary))

# Exclude cells with missing Diameter
complete_cells <- !is.na(meta[[COL_DIAMETER]])
meta     <- meta[complete_cells, ]
expr_mat <- expr_mat[, complete_cells]

cat(sprintf("\nCells with Diameter measurement : %d / %d  (%.1f%% retained)\n",
            nrow(meta), sum(!complete_cells) + nrow(meta),
            100 * mean(complete_cells)))
cat(sprintf("expr_mat dimensions             : %d proteins x %d cells\n",
            nrow(expr_mat), ncol(expr_mat)))

## 3. Protein Filtering

`fitExtractVarPartModel` requires each protein to be quantified in enough cells and across enough plates to estimate the random-effect variance components reliably. A protein that is only detected in one plate contributes no information to the plate random effect and will cause fitting errors.

Two thresholds control retention:

| Parameter | Default | Meaning |
|---|---|---|
| `min_cell_pct` | 25% | A protein must be detected in ≥ 25% of cells *within a plate* for that plate to count |
| `min_n_plates` | 5 | A protein must pass the cell-fraction threshold in at least 5 of the 9 plates |

The heatmap below shows the percentage of proteins retained across a grid of threshold combinations, helping to identify a threshold that balances stringency against proteome coverage.

### 3.1 Retention Heatmap

Each cell shows the percentage of proteins that would be retained at that combination of cell-fraction threshold (x-axis) and minimum number of plates (y-axis). Use this to validate that the chosen defaults (25%, 5 plates) sit in a reasonable region — not so loose that poorly-detected proteins dominate the model fit, not so strict that the proteome coverage drops below ~50%.

In [ ]:
plates   <- levels(meta[[COL_PLATE]])
n_plates <- length(plates)

# Per-protein, per-plate detection fraction
plate_comp_mat <- sapply(plates, function(plate) {
    idx    <- which(meta[[COL_PLATE]] == plate)
    m      <- expr_mat[, idx, drop = FALSE]
    comp   <- rowMeans(!is.na(m))
    comp[rowSums(!is.na(m)) == 0] <- NA   # protein absent from plate entirely
    comp
})  # n_proteins x n_plates

cell_thresholds  <- seq(0, 0.5, by = 0.025)
plate_thresholds <- seq_len(n_plates)

retention_grid <- outer(cell_thresholds, plate_thresholds, Vectorize(function(ct, pt) {
    passes <- plate_comp_mat >= ct
    mean(rowSums(passes, na.rm = TRUE) >= pt) * 100
}))

df_heatmap <- expand.grid(min_cells_pct = cell_thresholds * 100,
                           min_plates    = plate_thresholds) %>%
    mutate(retention = as.vector(retention_grid))

options(repr.plot.width = 12, repr.plot.height = 6)

ggplot(df_heatmap, aes(x = min_cells_pct, y = factor(min_plates), fill = retention)) +
    geom_tile(colour = "white", linewidth = 0.4) +
    geom_text(aes(label  = sprintf("%.0f%%", retention),
                  colour = retention > 50),
              size = 3) +
    scale_colour_manual(values = c("TRUE" = "black", "FALSE" = "white"), guide = "none") +
    scale_fill_distiller(palette = "Blues", direction = 1,
                         limits = c(0, 100), name = "% proteins\nretained") +
    scale_x_continuous(breaks = cell_thresholds * 100,
                       labels = paste0(cell_thresholds * 100, "%")) +
    theme_classic(base_size = 11) +
    labs(title    = "Protein retention vs. filtering thresholds",
         subtitle = sprintf("n = %d proteins, %d plates", nrow(expr_mat), n_plates),
         x        = "Min. detection in cells within plate",
         y        = "Min. number of plates") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          plot.title  = element_text(face = "bold"))

### 3.2 Apply Filter

Applies the chosen thresholds. Proteins that do not meet the criteria are removed from `expr_mat` for all downstream steps.

In [ ]:
filter_proteins <- function(expr_mat, meta, col_plate,
                             min_cell_pct = 0.25,
                             min_n_plates = 5) {
    plates     <- levels(meta[[col_plate]])
    plate_comp <- sapply(plates, function(plate) {
        idx  <- which(meta[[col_plate]] == plate)
        m    <- expr_mat[, idx, drop = FALSE]
        comp <- rowMeans(!is.na(m))
        comp[rowSums(!is.na(m)) == 0] <- NA
        comp
    })

    passes           <- plate_comp >= min_cell_pct
    n_plates_passing <- rowSums(passes, na.rm = TRUE)
    keep             <- n_plates_passing >= min_n_plates

    cat(sprintf("Protein filter:\n"))
    cat(sprintf("  Min. %% cells within plate : %.0f%%\n",   min_cell_pct * 100))
    cat(sprintf("  Min. number of plates     : %d / %d\n",  min_n_plates, length(plates)))
    cat(sprintf("  Proteins before           : %d\n",       nrow(expr_mat)))
    cat(sprintf("  Proteins after            : %d  (%.1f%% retained)\n",
                sum(keep), 100 * mean(keep)))

    expr_mat[keep, ]
}

MIN_CELL_PCT <- 0.25
MIN_N_PLATES <- 5

expr_mat <- filter_proteins(expr_mat, meta,
                             col_plate    = COL_PLATE,
                             min_cell_pct = MIN_CELL_PCT,
                             min_n_plates = MIN_N_PLATES)

## 4. Model Diagnostics — Injection Order Confounding

Before fitting, it is important to check whether injection order is confounded with cell type within plates. If K562 and HEK293T cells were consistently sorted to different injection positions (e.g., all K562 cells run first), then `injection_order` and `source name` are collinear and the model cannot separate their contributions.

**Two plots are shown:**
1. **Violin per plate** — distribution of z-scored injection order by cell type, faceted by plate. Overlapping distributions indicate no confounding.
2. **Mean injection order per cell type per plate** — a line connecting means across plates. A flat or criss-crossing pattern indicates weak confounding; a consistently separated pair of lines indicates strong confounding.

If confounding is detected, `injection_order` should be dropped from the formula or the affected plates flagged.

In [ ]:
cd_diag <- meta %>% rename(source_name = `source name`)

p_violin <- ggplot(cd_diag,
                   aes(x = source_name, y = injection_order_z, fill = source_name)) +
    geom_violin(scale = "width", alpha = 0.7, show.legend = FALSE) +
    geom_boxplot(width = 0.08, outlier.shape = NA, fill = "white", show.legend = FALSE) +
    facet_wrap(~ plate_id, ncol = 3) +
    scale_fill_brewer(palette = "Set1") +
    theme_classic(base_size = 10) +
    labs(title    = "Injection order by cell type — per plate",
         subtitle = "Overlapping distributions = no confounding",
         x = NULL, y = "Injection order (z-scored)") +
    theme(axis.text.x = element_text(angle = 35, hjust = 1),
          plot.title  = element_text(face = "bold"),
          strip.text  = element_text(face = "bold"))

p_mean <- cd_diag %>%
    group_by(plate_id, source_name) %>%
    summarise(mean_inj = mean(injection_order_z, na.rm = TRUE),
              se_inj   = sd(injection_order_z, na.rm = TRUE) / sqrt(n()),
              .groups  = "drop") %>%
    ggplot(aes(x = plate_id, y = mean_inj,
               colour = source_name, group = source_name)) +
    geom_line(linewidth = 0.8, alpha = 0.7) +
    geom_point(size = 2.5) +
    geom_errorbar(aes(ymin = mean_inj - se_inj, ymax = mean_inj + se_inj), width = 0.2) +
    geom_hline(yintercept = 0, linetype = "dashed", colour = "grey50") +
    scale_colour_brewer(palette = "Set1", name = "Cell type") +
    theme_classic(base_size = 11) +
    labs(title    = "Mean injection order per cell type per plate",
         subtitle = "Criss-crossing lines = weak confounding",
         x = "Plate", y = "Mean injection order (z-scored)") +
    theme(axis.text.x = element_text(angle = 35, hjust = 1),
          plot.title  = element_text(face = "bold"))

options(repr.plot.width = 14, repr.plot.height = 10)
print(p_violin)
options(repr.plot.width = 10, repr.plot.height = 4)
print(p_mean)

ggsave(file.path(FIG_OUT, "injection_order_confounding.png"),
       p_violin / p_mean, width = 14, height = 14)

## 5. Fitting Models

Fits three `variancePartition` models corresponding to successive correction stages. The workflow mirrors `01_scplainer.ipynb`: each stage regresses out a subset of technical effects, and the residuals are passed to the next model. This isolates how much variance each covariate group explains *on top of* effects already removed.

| Stage | Input matrix | Regressed out (via `lmFit`) | Model formula |
|---|---|---|---|
| `vp_fit` | Raw log₂ intensities | — | All effects |
| `vp_fit_norm` | Norm-corrected residuals | `log2_median_intensity + n_proteins` | Batch + biology |
| `vp_fit_batch` | Batch-corrected residuals | Norm + plate + position + injection | Biology only |

> **Runtime note** — each `fitExtractVarPartModel` call takes ~30–60 minutes on 64 cores. RDS files are written after each fit so later sections can be run independently.

### 5.1 Row-mean Imputation

`fitExtractVarPartModel` cannot handle missing values in the expression matrix. After protein filtering, each remaining `NA` (protein detected in some cells but not others) is replaced with the per-protein row mean. This preserves the mean expression level while filling gaps with the least informative value. Proteins that survived the filter are guaranteed to have at least one valid observation per plate (by construction), so row means are always well-defined.

In [ ]:
cat(sprintf("NAs before imputation: %d  (%.1f%%)\n",
            sum(is.na(expr_mat)), 100 * mean(is.na(expr_mat))))

expr_mat_impute <- t(apply(expr_mat, 1, function(x) {
    x[is.na(x)] <- mean(x, na.rm = TRUE)
    x
}))

cat(sprintf("NAs after  imputation: %d\n", sum(is.na(expr_mat_impute))))
cat(sprintf("Matrix dimensions    : %d proteins x %d cells\n",
            nrow(expr_mat_impute), ncol(expr_mat_impute)))

### 5.2 Model Formulas

Three formulas are defined, one per correction stage. Each formula only includes covariates that are *not* already regressed out in the input matrix, so that the variance fractions sum to ~100% and are interpretable.

| Formula | Stage | Covariates included |
|---|---|---|
| `vp_formula` | Full (no correction) | All normalisation + batch + biology terms |
| `vp_formula_norm` | Post-normalisation | Batch + biology (normalisation already removed) |
| `vp_formula_batch` | Post-batch | Biology only (`source name`, `Diameter`) |

**Design note** — in `variancePartition`, categorical variables enter as random effects `(1|var)` because the framework treats all grouping variables symmetrically. Continuous fixed effects (`Diameter_z`, `injection_order_z`, etc.) enter as standard fixed-effects terms. The nested random slope `(0 + X_pos | plate_id)` models a per-plate linear gradient in injection position without a plate-level intercept shift (the intercept is captured by `(1|plate_id)`).

In [ ]:
# Full model — all technical and biological effects
vp_formula <- as.formula(
    "~ (1 | plate_id) +
       (0 + `comment[X position]` | plate_id) +
       (1 | plate_id:date) +
       injection_order_z +
       (1 | `source name`) +
       Diameter_z +
       n_proteins_z +
       log2_median_intensity_z"
)

# After normalisation regression — drop n_proteins + log2_median_intensity terms
vp_formula_norm <- as.formula(
    "~ (1 | plate_id) +
       (0 + `comment[X position]` | plate_id) +
       (1 | plate_id:date) +
       injection_order_z +
       (1 | `source name`) +
       Diameter_z"
)

# After full batch regression — only biology remains
vp_formula_batch <- as.formula(
    "~ (1 | `source name`) +
       Diameter_z"
)

cat("Formulas defined.\n")
print(vp_formula)

### 5.3 Batch Correction via `lmFit`

Regresses out technical effects from `expr_mat_impute` using `limma::lmFit` before fitting the reduced-formula variancePartition models. Using `lmFit` residuals rather than simply dropping terms from the formula ensures that the regressed-out variance is fully removed, not just diluted, in the residual matrix fed to the next model.

| Residual | Regressors removed |
|---|---|
| `res_norm` | `log2_median_intensity + n_proteins` |
| `res_batch` | `plate_id + log2_median_intensity + n_proteins + X_position + injection_order` |

**Caution** — `lmFit` uses a fixed-effects model for correction, which assumes each plate is a fixed level rather than a sample from a random distribution. This differs from the random-effect treatment in `vp_formula`. The residuals are used solely as inputs to reduced variancePartition models; the full model (`vp_fit`) uses the uncorrected matrix.

In [ ]:
# Post-normalisation residuals
design_norm <- model.matrix(
    ~ log2_median_intensity + n_proteins,
    data = meta
)
fit_norm <- lmFit(expr_mat_impute, design_norm)
res_norm  <- residuals(fit_norm, expr_mat_impute)

# Post-batch residuals (normalisation + plate + position + injection order)
design_batch <- model.matrix(
    as.formula(paste0(
        "~ plate_id + log2_median_intensity + n_proteins + `comment[X position]` + injection_order_z"
    )),
    data = meta
)
fit_batch <- lmFit(expr_mat_impute, design_batch)
res_batch  <- residuals(fit_batch, expr_mat_impute)

cat(sprintf("res_norm  : %d x %d  (NAs: %d)\n",
            nrow(res_norm),  ncol(res_norm),  sum(is.na(res_norm))))
cat(sprintf("res_batch : %d x %d  (NAs: %d)\n",
            nrow(res_batch), ncol(res_batch), sum(is.na(res_batch))))

### 5.4 Fit variancePartition Models

Fits three models via `fitExtractVarPartModel`. Each returns a `varPartResults` data frame (proteins × covariates) giving the percentage of variance explained by each term. Models are saved to disk and results exported to CSV for easy inspection outside R.

In [ ]:
cat("[1/3] Fitting full model...\n")
vp_fit <- fitExtractVarPartModel(
    exprObj = expr_mat_impute,
    formula = vp_formula,
    data    = meta,
    BPPARAM = bpparam()
)
saveRDS(vp_fit, file.path(MODELS_OUT, "vp_fit.rds"))
write.csv(as.data.frame(vp_fit),
          file.path(MODELS_OUT, "variance_fractions_full.csv"),
          row.names = TRUE)
cat("  Saved: vp_fit.rds\n\n")

cat("[2/3] Fitting post-normalisation model...\n")
vp_fit_norm <- fitExtractVarPartModel(
    exprObj = res_norm,
    formula = vp_formula_norm,
    data    = meta,
    BPPARAM = bpparam()
)
saveRDS(vp_fit_norm, file.path(MODELS_OUT, "vp_fit_norm.rds"))
cat("  Saved: vp_fit_norm.rds\n\n")

cat("[3/3] Fitting post-batch model...\n")
vp_fit_batch <- fitExtractVarPartModel(
    exprObj = res_batch,
    formula = vp_formula_batch,
    data    = meta,
    BPPARAM = bpparam()
)
saveRDS(vp_fit_batch, file.path(MODELS_OUT, "vp_fit_batch.rds"))
cat("  Saved: vp_fit_batch.rds\n")

## 6. Variance Distributions

`plotVarPart` (from `variancePartition`) draws a violin plot per covariate showing the distribution of per-protein variance fractions across all retained proteins. Covariates are sorted by median variance explained (highest first).

**Interpreting the progression across stages:**
- **Full model** — shows the raw decomposition. Expect normalisation terms (`log2_median_intensity`, `n_proteins`) and `plate_id` to dominate. Residual should be large if many unexplained sources remain.
- **Post-normalisation** — normalisation terms are removed; batch and biology terms should grow proportionally to fill the gap.
- **Post-batch** — only `source name` and `Diameter` remain; their combined fraction indicates how much of the variance is biological after full technical correction. A high residual here means unexplained cell-to-cell heterogeneity.

In [ ]:
if (!exists("vp_fit"))       vp_fit       <- readRDS(file.path(MODELS_OUT, "vp_fit.rds"))
if (!exists("vp_fit_norm"))  vp_fit_norm  <- readRDS(file.path(MODELS_OUT, "vp_fit_norm.rds"))
if (!exists("vp_fit_batch")) vp_fit_batch <- readRDS(file.path(MODELS_OUT, "vp_fit_batch.rds"))

options(repr.plot.width = 12, repr.plot.height = 5)

p1 <- plotVarPart(sortCols(vp_fit))       + ggtitle("Full model")
p2 <- plotVarPart(sortCols(vp_fit_norm))  + ggtitle("Post-normalisation")
p3 <- plotVarPart(sortCols(vp_fit_batch)) + ggtitle("Post-batch correction")

ggsave(file.path(FIG_OUT, "varpart_full.png"),  p1, width = 12, height = 5)
ggsave(file.path(FIG_OUT, "varpart_norm.png"),  p2, width = 12, height = 5)
ggsave(file.path(FIG_OUT, "varpart_batch.png"), p3, width = 12, height = 5)

print(p1)
print(p2)
print(p3)

## 7. Dimensionality Reduction

PCA on the corrected expression matrices provides a complementary visual validation of the variance decomposition. Three matrices are plotted:

| Matrix | What it shows |
|---|---|
| `expr_mat_impute` | Raw structure — expected to cluster by plate and instrument |
| `res_norm` | After normalisation correction — intensity-scale batch reduced |
| `res_batch` | After full batch correction — should show separation by cell type only |

Each matrix yields three plots coloured by `plate_id`, `source name`, and `comment[X position]` respectively. `prcomp` is applied after removing proteins with any remaining `NA` (in practice none after imputation).

In [ ]:
plot_pca <- function(mat, meta, colour_var, shape_var = NULL, title = "") {
    complete  <- rowSums(is.na(mat)) == 0
    pca       <- prcomp(t(mat[complete, ]), scale. = TRUE, center = TRUE)
    var_expl  <- summary(pca)$importance[2, 1:2] * 100

    df <- data.frame(PC1 = pca$x[, 1], PC2 = pca$x[, 2], group = meta[[colour_var]])
    is_cont <- is.numeric(meta[[colour_var]])
    if (!is_cont)              df$group <- factor(df$group)
    if (!is.null(shape_var))   df$shape <- factor(meta[[shape_var]])

    p <- ggplot(df, aes(x = PC1, y = PC2, colour = group)) +
        { if (!is.null(shape_var)) geom_point(aes(shape = shape), alpha = 0.7, size = 1.8)
          else                     geom_point(alpha = 0.7, size = 1.8) } +
        { if (is_cont)
            scale_colour_distiller(palette = "RdYlBu", name = colour_var)
          else {
            n  <- nlevels(factor(df$group))
            pal <- colorRampPalette(brewer.pal(min(n, 9), "Set1"))(n)
            scale_colour_manual(values = pal, name = colour_var)
          }
        } +
        { if (!is.null(shape_var))
            scale_shape_manual(values = seq_len(nlevels(factor(meta[[shape_var]]))),
                               name = shape_var)
        } +
        theme_classic(base_size = 12) +
        labs(title = title,
             x = sprintf("PC1 (%.1f%%)", var_expl[1]),
             y = sprintf("PC2 (%.1f%%)", var_expl[2])) +
        theme(plot.title = element_text(face = "bold"))
    p
}

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

for (stage in list(
    list(mat = expr_mat_impute, label = "Raw"),
    list(mat = res_norm,        label = "Post-normalisation"),
    list(mat = res_batch,       label = "Post-batch correction")
)) {
    p1 <- plot_pca(stage$mat, meta,
                   colour_var = COL_PLATE,
                   shape_var  = COL_CELL_TYPE,
                   title      = sprintf("%s — by plate", stage$label))
    p2 <- plot_pca(stage$mat, meta,
                   colour_var = COL_CELL_TYPE,
                   title      = sprintf("%s — by cell type", stage$label))
    p3 <- plot_pca(stage$mat, meta,
                   colour_var = COL_X_POSITION,
                   shape_var  = COL_CELL_TYPE,
                   title      = sprintf("%s — by X position", stage$label))

    combined <- p1 | p2 | p3
    ggsave(file.path(FIG_OUT, sprintf("pca_%s.png", tolower(gsub(" ", "_", stage$label)))),
           combined, width = 14, height = 5)
    print(combined)
}

## 8. Export

Saves the variance fraction matrices as CSV files for use in downstream analyses or figure generation outside R. The CSV files complement the RDS objects: they are human-readable and can be loaded in Python without an R runtime.

In [ ]:
if (!exists("vp_fit"))       vp_fit       <- readRDS(file.path(MODELS_OUT, "vp_fit.rds"))
if (!exists("vp_fit_norm"))  vp_fit_norm  <- readRDS(file.path(MODELS_OUT, "vp_fit_norm.rds"))
if (!exists("vp_fit_batch")) vp_fit_batch <- readRDS(file.path(MODELS_OUT, "vp_fit_batch.rds"))

write.csv(as.data.frame(vp_fit),
          file.path(MODELS_OUT, "variance_fractions_full.csv"),  row.names = TRUE)
write.csv(as.data.frame(vp_fit_norm),
          file.path(MODELS_OUT, "variance_fractions_norm.csv"),  row.names = TRUE)
write.csv(as.data.frame(vp_fit_batch),
          file.path(MODELS_OUT, "variance_fractions_batch.csv"), row.names = TRUE)

cat("Exported:\n")
for (f in c("variance_fractions_full.csv",
            "variance_fractions_norm.csv",
            "variance_fractions_batch.csv")) {
    cat(sprintf("  %s\n", file.path(MODELS_OUT, f)))
}